# eco
Interactive namespace dashboard. Rendered as a live widget by Voila, or opened as a normal notebook in JupyterLab.
The scope is chosen by the `eco` launcher via the `ECO_SCOPE` environment variable (default `bernina`).

Pick a component and press **Open** to show its widget. Each opened widget can be collapsed or closed (X) individually; closing fully stops its refresh so a long session stays light. Switch **Panels / Master-detail** to change how many stay open at once.

In [ ]:
import os
import importlib
from IPython.display import display, Markdown

scope = os.environ.get("ECO_SCOPE") or "bernina"
lazy = os.environ.get("ECO_LAZY", "1") not in ("0", "false", "False", "")
layout = os.environ.get("ECO_LAYOUT", "panels")  # panels (default) | detail

display(Markdown(f"## {scope}" + ("  \u00b7  *lazy*" if lazy else "")))

# Configure lazy initialisation before importing the scope, exactly like the
# shell startup does.
from eco import ecocnf
if lazy:
    ecocnf.startup_lazy = True

mod = importlib.import_module(f"eco.{scope}")
# Each scope module exposes a `namespace` root; fall back to the module itself.
target = getattr(mod, "namespace", mod)

In [ ]:
# Primary: the managed dashboard - open sub-component widgets on demand into a
# tray you can collapse / close, so a pure-Voila session never accumulates an
# unbounded pile of live widgets. Falls back to the assembly browser, then the
# object's own widget / rich display.
shown = False
try:
    from eco.widgets.widget_tray import make_namespace_dashboard
    display(make_namespace_dashboard(target, mode=layout))
    shown = True
except Exception as e:
    print(f"dashboard unavailable: {e}")

if not shown:
    try:
        from eco.widgets.assembly_browser import show_assembly_browser
        display(show_assembly_browser(target))
        shown = True
    except Exception as e:
        print(f"assembly browser unavailable: {e}")

if not shown:
    try:
        display(target.widget())
        shown = True
    except Exception as e:
        print(f"widget() unavailable: {e}")

if not shown:
    display(target)